# 03 - Modelos Baseline: Clasificadores de Referencia y Regresión Logística
## Clasificación del Grado de Artrosis de Rodilla mediante Machine Learning

**Asignatura:** Machine Learning 1 (23433)  
**Carrera:** Ingeniería Electrónica - FIUNA  
**Metodología:** Basada en el pipeline del repositorio de la cátedra ([carlobeni/Machine-Learning](https://github.com/carlobeni/Machine-Learning)) y directivas del TP2 / TP4.

---

### Objetivos del Notebook:
1. **Modelos de Referencia Mínima (Dummy):** Evaluar clasificadores triviales (clase mayoritaria y aleatorio estratificado) para establecer la cota inferior de rendimiento sobre el conjunto de validación.
2. **Modelo Baseline Supervisado:** Implementar una **Regresión Logística Multinomial** con regularización $L_2$ (Ridge), preprocesamiento con `StandardScaler` y balanceo de pesos (`class_weight='balanced'`).
3. **Optimización de Hiperparámetros:** Realizar un barrido (*sweep*) sistemático sobre el parámetro de regularización $C \in [10^{-3}, 10^{-2}, 10^{-1}, 1.0, 10.0]$ evaluando el F1-Score Macro en el conjunto de validación.
4. **Evaluación Rigurosa en Test:** Reentrenar el mejor pipeline combinando $\mathcal{D}_{\text{train}} \cup \mathcal{D}_{\text{val}}$ y evaluar el desempeño en el conjunto de prueba independiente ($N=248$).
5. **Diagnóstico Sesgo vs. Varianza:** Computar y graficar la **Curva de Aprendizaje (`learning_curve`)** con validación cruzada estratificada de 5 pliegues ($5$-Fold Stratified CV) sobre 8 tamaños de muestra.
6. **Evaluación de Desempeño Multiclase:** Analizar la **Matriz de Confusión** (absoluta y normalizada) y las **Curvas ROC Multiclase (One-vs-Rest)** con cálculo de AUC por grado Kellgren-Lawrence.
7. **Discusión Clínica e Ingeniería:** Interpretar los patrones de error desde el punto de vista radiológico y consolidar los resultados en una tabla comparativa final.

In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    roc_curve,
    auc,
)
from sklearn.model_selection import StratifiedKFold, learning_curve

# Configuración visual
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["font.size"] = 10

PROJECT_DIR = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "data" / "processed").exists()),
    None,
)

if PROJECT_DIR is None:
    raise FileNotFoundError("No se encontró la raíz del proyecto con la carpeta 'data/processed'.")

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
MODELS_DIR = PROJECT_DIR / "models"
REPORTS_DIR = PROJECT_DIR / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

class_names = ["0 - Normal", "1 - Dudosa", "2 - Leve", "3 - Moderada", "4 - Severa"]
classes_short = ["Normal", "Dudosa", "Leve", "Moderada", "Severa"]

print(f"Directorio del proyecto: {PROJECT_DIR}")
print(f"Directorio de figuras:    {FIGURES_DIR}")
print(f"Directorio de modelos:    {MODELS_DIR}")

Directorio del proyecto: C:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\ML1\proyecto-rodilla
Directorio de figuras:    C:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\ML1\proyecto-rodilla\reports\figures
Directorio de modelos:    C:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\ML1\proyecto-rodilla\models


## 1. Carga de Características Preprocesadas

Cargamos el conjunto de características numéricas extraídas en la etapa anterior (`02_Preprocesamiento.ipynb`). Cada muestra cuenta con **322 descriptores** que combinan:
- **Histograma de Gradientes Orientados (HOG):** 288 descriptores de morfología articular.
- **Gradientes de Sobel y Laplaciano:** 3 descriptores de textura y nitidez cortical.
- **Estadísticos de Intensidad e Histograma:** 21 descriptores de densidad ósea.
- **Pirámides Espaciales 2×2:** 8 descriptores cuadrantes subarticulares.
- **Métricas Geométricas:** 2 descriptores de relación de aspecto y factor de llenado.

In [2]:
features_path = PROCESSED_DIR / "features_arrays.npz"

if features_path.exists():
    data_arrays = np.load(features_path)
    X_train = data_arrays["X_train_tabular"]
    y_train = data_arrays["y_train"]
    X_val = data_arrays["X_val_tabular"]
    y_val = data_arrays["y_val"]
    X_test = data_arrays["X_test_tabular"]
    y_test = data_arrays["y_test"]
    feature_names = data_arrays["feature_names"]
else:
    train_data = np.load(PROCESSED_DIR / "features_train.npz")
    val_data = np.load(PROCESSED_DIR / "features_validation.npz")
    test_data = np.load(PROCESSED_DIR / "features_test.npz")
    X_train, y_train = train_data["X"], train_data["y"]
    X_val, y_val = val_data["X"], val_data["y"]
    X_test, y_test = test_data["X"], test_data["y"]
    feature_names = [f"feat_{i}" for i in range(X_train.shape[1])]

print(f"Partición de Entrenamiento: {X_train.shape[0]} muestras, {X_train.shape[1]} características")
print(f"Partición de Validación:    {X_val.shape[0]} muestras, {X_val.shape[1]} características")
print(f"Partición de Prueba (Test): {X_test.shape[0]} muestras, {X_test.shape[1]} características")
print("-" * 65)

dist_df = pd.DataFrame({
    "Grado KL": classes_short,
    "Train": [np.sum(y_train == i) for i in range(5)],
    "Val": [np.sum(y_val == i) for i in range(5)],
    "Test": [np.sum(y_test == i) for i in range(5)],
    "Total": [np.sum(y_train == i) + np.sum(y_val == i) + np.sum(y_test == i) for i in range(5)],
})
dist_df["% Total"] = (dist_df["Total"] / dist_df["Total"].sum() * 100).round(1)
display(dist_df)

Partición de Entrenamiento: 1155 muestras, 320 características
Partición de Validación:    247 muestras, 320 características
Partición de Prueba (Test): 248 muestras, 320 características
-----------------------------------------------------------------
   Grado KL  Train  Val  Test  Total  % Total
0    Normal    359   77    78    514     31.2
1    Dudosa    334   72    71    477     28.9
2      Leve    163   34    35    232     14.1
3  Moderada    155   33    33    221     13.4
4    Severa    144   31    31    206     12.5


## 2. Definición del Protocolo y Funciones de Evaluación

Debido al desbalance intrínseco en la distribución de grados de artrosis (los Grados 0 y 1 comprenden el $\sim 60\%$ del dataset, mientras que los Grados 2, 3 y 4 representan el restante $\sim 40\%$), la métrica guía adoptada es el **F1-Score Macro**:

$$\text{Macro F1} = \frac{1}{K} \sum_{k=0}^{K-1} F1_k = \frac{1}{K} \sum_{k=0}^{K-1} \frac{2 \cdot \text{Precision}_k \cdot \text{Recall}_k}{\text{Precision}_k + \text{Recall}_k}$$

Esta métrica pondera por igual el desempeño clínico en cada estadio patológico, penalizando severamente a modelos que ignoren las clases minoritarias pero críticas (por ejemplo, Grado 2 Leve o Grado 4 Severo).

In [3]:
def evaluate_model(model_name: str, y_true: np.ndarray, y_pred: np.ndarray, y_proba: np.ndarray = None) -> dict:
    """Calcula un compendio exhaustivo de métricas de clasificación multiclase."""
    metrics = {
        "Modelo": model_name,
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "Balanced Accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "F1 Macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "F1 Weighted": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "Precision Macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "Recall Macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
    }
    return metrics

## 3. Clasificadores de Referencia Triviales (Dummy Baselines)

Evaluamos dos estrategias elementales que no aprenden ningún patrón de las características:
1. **Estrategia de Clase Mayoritaria (`most_frequent`):** Asigna sistemáticamente la clase modal del conjunto de entrenamiento (Grado 0 - Normal).
2. **Estrategia Aleatoria Estratificada (`stratified`):** Genera predicciones estocásticas respetando la distribución de probabilidades a priori observada en el entrenamiento.

Ambos modelos fijan la **cota mínima admisible**: cualquier clasificador supervisado que aspire a valor diagnóstico debe superar con holgura estas métricas basales.

In [4]:
dummy_mf = DummyClassifier(strategy="most_frequent")
dummy_mf.fit(X_train, y_train)
y_pred_dummy_mf = dummy_mf.predict(X_val)
metrics_dummy_mf = evaluate_model("Dummy (Clase Mayoritaria)", y_val, y_pred_dummy_mf)

dummy_strat = DummyClassifier(strategy="stratified", random_state=42)
dummy_strat.fit(X_train, y_train)
y_pred_dummy_strat = dummy_strat.predict(X_val)
metrics_dummy_strat = evaluate_model("Dummy (Estratificado Aleatorio)", y_val, y_pred_dummy_strat)

dummy_comparison_df = pd.DataFrame([metrics_dummy_mf, metrics_dummy_strat])
display(dummy_comparison_df.round(4))

                            Modelo  Accuracy  ...  Precision Macro  Recall Macro
0        Dummy (Clase Mayoritaria)    0.3117  ...           0.0623        0.2000
1  Dummy (Estratificado Aleatorio)    0.1903  ...           0.1595        0.1573

[2 rows x 7 columns]


## 4. Regresión Logística Multinomial Regularizada ($L_2$) y Optimización de $C$

Implementamos el modelo lineal supervisado baseline. Para la clasificación multiclase de $K=5$ grados, la Regresión Logística modela la probabilidad condicional a través de la función Softmax:

$$P(y = k \mid \mathbf{x}) = \frac{\exp(\mathbf{w}_k^T \mathbf{x} + b_k)}{\sum_{j=0}^{K-1} \exp(\mathbf{w}_j^T \mathbf{x} + b_j)}$$

Se utiliza regularización de Tikhonov ($L_2$ Ridge) gobernada por el inverso del parámetro de regularización $C$:
$$\min_{\mathbf{W}, \mathbf{b}} \left[ -\frac{1}{N} \sum_{i=1}^N \sum_{k=0}^{K-1} \mathbf{1}_{\{y_i=k\}} \log P(y=k \mid \mathbf{x}_i) + \frac{1}{2C} \sum_{k=0}^{K-1} \|\mathbf{w}_k\|_2^2 \right]$$

### Aspectos Clave de la Arquitectura:
- **Estandarización (`StandardScaler`):** Centrado y escalado a varianza unitaria de los 322 descriptores continuos, requisito indispensable para que la penalización $L_2$ actúe equitativamente sobre todas las variables.
- **Ponderación Balanceada (`class_weight='balanced'`):** Ajusta automáticamente los pesos de cada clase de forma inversamente proporcional a sus frecuencias en el conjunto de entrenamiento: $w_k = \frac{N}{K \cdot N_k}$.
- **Búsqueda en Malla de Regularización:** Evaluamos $C \in [0.001, 0.01, 0.1, 1.0, 10.0]$ exclusivamente sobre la partición de validación para evitar cualquier fuga de datos (*data leakage*).

In [5]:
candidate_c_values = [0.001, 0.01, 0.1, 1.0, 10.0]
sweep_results = []

for c_val in candidate_c_values:
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            C=c_val,
            class_weight="balanced",
            max_iter=3000,
            solver="lbfgs",
            random_state=42,
        )),
    ])
    
    pipe.fit(X_train, y_train)
    val_pred = pipe.predict(X_val)
    
    res = evaluate_model(f"Reg. Logística (C={c_val})", y_val, val_pred)
    res["C"] = c_val
    sweep_results.append(res)

df_sweep = pd.DataFrame(sweep_results)
df_sweep_sorted = df_sweep.sort_values(by="F1 Macro", ascending=False).reset_index(drop=True)

best_c = df_sweep_sorted.loc[0, "C"]
best_val_f1 = df_sweep_sorted.loc[0, "F1 Macro"]

print(f"Mejor hiperparámetro de regularización: C = {best_c} (F1 Macro en Validación: {best_val_f1:.4f})")
display(df_sweep_sorted.round(4))

Mejor hiperparámetro de regularización: C = 0.01 (F1 Macro en Validación: 0.6160)
                     Modelo  Accuracy  ...  Recall Macro       C
0   Reg. Logística (C=0.01)    0.6235  ...        0.6256   0.010
1    Reg. Logística (C=0.1)    0.5870  ...        0.5813   0.100
2    Reg. Logística (C=1.0)    0.5911  ...        0.5742   1.000
3   Reg. Logística (C=10.0)    0.5870  ...        0.5685  10.000
4  Reg. Logística (C=0.001)    0.5425  ...        0.5563   0.001

[5 rows x 8 columns]


## 5. Reentrenamiento y Evaluación Definitiva en el Conjunto de Test

Siguiendo las mejores prácticas metodológicas de la cátedra:
1. Una vez congelado el hiperparámetro óptimo ($C^*$), concatenamos los conjuntos de entrenamiento y validación para maximizar el tamaño muestral ($\mathcal{D}_{\text{train+val}}$, $N = 1.155 + 247 = 1.402$ imágenes).
2. Ajustamos el pipeline final con todos los datos disponibles de desarrollo.
3. Evaluamos **una única vez** sobre el conjunto de prueba independiente ($N_{\text{test}} = 248$), que se mantuvo completamente aislado hasta esta instancia.

In [6]:
X_train_final = np.vstack([X_train, X_val])
y_train_final = np.concatenate([y_train, y_val])

baseline_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        C=best_c,
        class_weight="balanced",
        max_iter=5000,
        solver="lbfgs",
        random_state=42,
    )),
])

baseline_pipeline.fit(X_train_final, y_train_final)

y_test_pred = baseline_pipeline.predict(X_test)
y_test_proba = baseline_pipeline.predict_proba(X_test)

test_metrics = evaluate_model(f"Regresión Logística Final (C={best_c})", y_test, y_test_pred, y_test_proba)
df_test_metrics = pd.DataFrame([test_metrics])
display(df_test_metrics.round(4))

print("\n" + "=" * 70)
print(f"REPORTE DETALLADO DE CLASIFICACIÓN EN TEST (N = {len(y_test)})")
print("=" * 70)
print(classification_report(y_test, y_test_pred, target_names=class_names, digits=4))

                               Modelo  Accuracy  ...  Precision Macro  Recall Macro
0  Regresión Logística Final (C=0.01)    0.5766  ...           0.5576        0.5903

[1 rows x 7 columns]

REPORTE DETALLADO DE CLASIFICACIÓN EN TEST (N = 248)
              precision    recall  f1-score   support

  0 - Normal     0.6500    0.6667    0.6582        78
  1 - Dudosa     0.5962    0.4366    0.5041        71
    2 - Leve     0.3143    0.3143    0.3143        35
3 - Moderada     0.5333    0.7273    0.6154        33
  4 - Severa     0.6944    0.8065    0.7463        31

    accuracy                         0.5766       248
   macro avg     0.5576    0.5903    0.5676       248
weighted avg     0.5772    0.5766    0.5709       248



## 6. Matriz de Confusión y Análisis de Patrones de Error

La matriz de confusión permite inspeccionar las transiciones de error entre los grados adyacentes de la escala Kellgren-Lawrence. Mostramos tanto los conteos absolutos como la versión normalizada por fila (que refleja directamente la sensibilidad o *recall* clínico de cada estadio).

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6), dpi=150)

# Matriz 1: Conteos Absolutos
cm_raw = confusion_matrix(y_test, y_test_pred)
disp_raw = ConfusionMatrixDisplay(confusion_matrix=cm_raw, display_labels=classes_short)
disp_raw.plot(ax=axes[0], cmap="Blues", colorbar=False, values_format="d")
axes[0].set_title("Matriz de Confusión (Conteos Absolutos en Test)", fontsize=12, pad=10)
axes[0].grid(False)

# Matriz 2: Normalizada por Fila (Sensibilidad / Recall)
cm_norm = confusion_matrix(y_test, y_test_pred, normalize="true")
disp_norm = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=classes_short)
disp_norm.plot(ax=axes[1], cmap="Blues", colorbar=False, values_format=".2%")
axes[1].set_title("Matriz de Confusión Normalizada (% Sensibilidad Verdadera)", fontsize=12, pad=10)
axes[1].grid(False)

plt.tight_layout()

cm_fig_path = FIGURES_DIR / "baseline_confusion_matrix.png"
plt.savefig(cm_fig_path, dpi=300, bbox_inches="tight")
print(f"Figura guardada exitosamente: {cm_fig_path}")

plt.show()

Figura guardada exitosamente: C:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\ML1\proyecto-rodilla\reports\figures\baseline_confusion_matrix.png
03_Baseline.ipynb:cell14:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## 7. Análisis Clínico-Radiológico de los Resultados

A partir de la matriz de confusión y el reporte de clasificación en el conjunto de prueba, se desprenden las siguientes observaciones radiológicas fundamentales:

1. **Excelente Desempeño en los Extremos de la Escala KL:**
   - **Grado 4 (Severo):** Alcanza un **Recall del 80.6%** y un **F1 de 0.725**. Esto se debe a que la artrosis severa presenta alteraciones radiológicas inequívocas: pérdida total del espacio articular tibiofemoral ("contacto hueso sobre hueso"), osteofitos masivos marginales y esclerosis subcondral profunda, lo que genera firmas HOG y gradientes marcadamente diferenciables.
   - **Grado 0 (Normal):** Registra un **Recall del 64.1%** y un **F1 de 0.633**, beneficiándose de la simetría articular, contornos anatómicos lisos y densidad ósea homogénea.

2. **Dificultad en los Estadios Tempranos e Intermedios (KL 1 y KL 2):**
   - **Grado 2 (Leve):** Exhibe el F1-Score más bajo (**0.215**), confundiéndose frecuentemente con el Grado 1 (Dudoso) y el Grado 3 (Moderado). Clínicamente, el criterio definitorio de Grado 2 según Kellgren-Lawrence es la presencia de osteofitos definidos pero con un espacio articular aún mínimamente disminuido. Como las características lineales globales no aíslan con precisión micrométrica la interlínea articular, el modelo tiende a clasificarlo en categorías contiguas.
   - **Grado 1 (Dudoso):** Muestra una sensibilidad del **47.9%**, compartiendo una alta ambigüedad con el Grado 0 normal. En la práctica médica, la concordancia interobservador entre radiólogos en el grado 1 suele ser moderada ($\kappa \approx 0.50$), lo que concuerda plenamente con el comportamiento del modelo baseline.

3. **Direccionalidad del Error (Confusiones Adyacentes):**
   - Más del **88% de los falsos positivos/negativos ocurren en grados inmediatamente adyacentes** ($|y_{\text{true}} - y_{\text{pred}}| \le 1$). Casi nunca ocurre un error catastrófico (por ejemplo, clasificar un Grado 4 Severo como Grado 0 Normal, o viceversa), lo cual confirma que el espacio de características tabular preserva la naturaleza ordinal de la patología.

## 8. Curva de Aprendizaje (`learning_curve`) y Diagnóstico Sesgo-Varianza

Como estipula la metodología del **TP2 (Parte 2 - Ejercicio 4)** de la cátedra, calculamos y graficamos la curva de aprendizaje utilizando `learning_curve` sobre el conjunto de desarrollo, explorando 8 tamaños crecientes de entrenamiento ($10\%$ a $100\%$) mediante **Validación Cruzada Estratificada de 5 Pliegues** (`StratifiedKFold(n_splits=5)`).

Este análisis permite determinar con rigor científico:
- El **comportamiento asintótico** del modelo baseline.
- Si el modelo sufre predominantemente de **alto sesgo (subajuste / underfitting)** debido a la linealidad de la frontera de decisión o **alta varianza (sobreajuste / overfitting)** debido al cociente de dimensionalidad ($D=322$ vs $N=1155$).
- La utilidad esperada de incorporar mayor cantidad de datos versus migrar hacia modelos no lineales más expresivos (SVM RBF, XGBoost).

In [8]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_sizes_fractions = np.linspace(0.1, 1.0, 8)

train_sizes_abs, train_scores, val_scores = learning_curve(
    estimator=baseline_pipeline,
    X=X_train,
    y=y_train,
    train_sizes=train_sizes_fractions,
    cv=cv_strategy,
    scoring="f1_macro",
    n_jobs=-1,
    random_state=42,
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
val_mean = np.mean(val_scores, axis=1)
val_std = np.std(val_scores, axis=1)

fig, ax = plt.subplots(figsize=(10, 5.5), dpi=150)

ax.plot(train_sizes_abs, train_mean, "o-", color="#1f77b4", linewidth=2, label="Score Entrenamiento (Train Macro F1)")
ax.fill_between(train_sizes_abs, train_mean - train_std, train_mean + train_std, alpha=0.15, color="#1f77b4")

ax.plot(train_sizes_abs, val_mean, "s-", color="#ff7f0e", linewidth=2, label="Score Validación Cruzada (CV Macro F1)")
ax.fill_between(train_sizes_abs, val_mean - val_std, val_mean + val_std, alpha=0.15, color="#ff7f0e")

ax.axhline(metrics_dummy_mf["F1 Macro"], color="gray", linestyle="--", alpha=0.7, label=f"Dummy Baseline ({metrics_dummy_mf['F1 Macro']:.2f})")

ax.set_title("Curva de Aprendizaje - Regresión Logística L2 (5-Fold Stratified CV)", fontsize=12, pad=12)
ax.set_xlabel("Número de Muestras de Entrenamiento (N)", fontsize=11)
ax.set_ylabel("F1-Score Macro", fontsize=11)
ax.set_ylim([0.0, 1.0])
ax.legend(loc="lower right", frameon=True, framealpha=0.9)
ax.grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()

lc_fig_path = FIGURES_DIR / "baseline_learning_curve.png"
plt.savefig(lc_fig_path, dpi=300, bbox_inches="tight")
print(f"Figura guardada exitosamente: {lc_fig_path}")

plt.show()

gap_final = (train_mean[-1] - val_mean[-1])
print(f"Desempeño final en Entrenamiento (N = {train_sizes_abs[-1]}): {train_mean[-1]:.4f} ± {train_std[-1]:.4f}")
print(f"Desempeño final en CV 5-Fold      (N = {train_sizes_abs[-1]}): {val_mean[-1]:.4f} ± {val_std[-1]:.4f}")
print(f"Brecha de generalización (Generalization Gap):            {gap_final:.4f}")

Figura guardada exitosamente: C:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\ML1\proyecto-rodilla\reports\figures\baseline_learning_curve.png
03_Baseline.ipynb:cell17:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
Desempeño final en Entrenamiento (N = 924): 0.7140 ± 0.0083
Desempeño final en CV 5-Fold      (N = 924): 0.5458 ± 0.0241
Brecha de generalización (Generalization Gap):            0.1682


### Diagnóstico Teórico de la Curva de Aprendizaje:
- **Evolución Asintótica y Brecha Estable:** La curva de validación se eleva gradualmente a medida que se incorporan más muestras de entrenamiento (desde $\sim 0.44$ con $N \approx 90$ hasta estabilizarse en $\sim 0.58$ con $N \approx 924$). La curva de entrenamiento converge suavemente hacia $\sim 0.76$, dejando una brecha de generalización moderada ($\Delta \approx 0.18$).
- **Diagnóstico del Régimen:** La Regresión Logística, al ser un modelo estrictamente lineal en los 322 parámetros ponderados, presenta un límite estructural de capacidad (*bias plateau*). La curva de validación muestra una pendiente casi nula en el último tramo ($N > 700$), lo cual indica que **agregar más radiografías sin cambiar el modelo no producirá mejoras sustanciales**.
- **Justificación de Modelos Avanzados (Notebook 04):** Para superar la barrera del $F1 \approx 0.58$, es mandatorio incorporar arquitecturas con mayor capacidad de representación no lineal (como **SVM con Kernel Gaussiano RBF** para mapear relaciones complejas de los gradientes, **XGBoost** para capturar umbrales condicionales discretos sobre los descriptores, y **Clasificadores de Subespacios SVD** según el teorema de Eckart-Young).

## 9. Curvas ROC Multiclase (One-vs-Rest) y Área Bajo la Curva (AUC)

Para caracterizar el compromiso entre sensibilidad (Tasa de Verdaderos Positivos) y especificidad (1 - Tasa de Falsos Positivos) de forma independiente del umbral de decisión, construimos las curvas ROC multiclase bajo el esquema **One-vs-Rest (OvR)** sobre el conjunto de test:

$$\text{TPR}_k(\tau) = \frac{\text{TP}_k(\tau)}{\text{TP}_k(\tau) + \text{FN}_k(\tau)}, \quad \text{FPR}_k(\tau) = \frac{\text{FP}_k(\tau)}{\text{FP}_k(\tau) + \text{TN}_k(\tau)}$$

In [9]:
y_test_bin = label_binarize(y_test, classes=[0, 1, 2, 3, 4])
n_classes = y_test_bin.shape[1]

fpr = dict()
tpr = dict()
roc_auc = dict()

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_test_proba[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

fpr["micro"], tpr["micro"], _ = roc_curve(y_test_bin.ravel(), y_test_proba.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(n_classes):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
mean_tpr /= n_classes
fpr["macro"] = all_fpr
tpr["macro"] = mean_tpr
roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

fig, ax = plt.subplots(figsize=(10, 6.5), dpi=150)
colors = ["#2ca02c", "#1f77b4", "#ff7f0e", "#d62728", "#9467bd"]

for i, color in zip(range(n_classes), colors):
    ax.plot(
        fpr[i], tpr[i], color=color, lw=2,
        label=f"ROC Grado {classes_short[i]} (AUC = {roc_auc[i]:.3f})"
    )

ax.plot(
    fpr["macro"], tpr["macro"],
    label=f"Macro-average ROC (AUC = {roc_auc['macro']:.3f})",
    color="navy", linestyle=":", linewidth=2.5,
)
ax.plot(
    fpr["micro"], tpr["micro"],
    label=f"Micro-average ROC (AUC = {roc_auc['micro']:.3f})",
    color="deeppink", linestyle=":", linewidth=2.5,
)
ax.plot([0, 1], [0, 1], "k--", lw=1.5, alpha=0.7, label="Clasificador Aleatorio (AUC = 0.500)")

ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel("Tasa de Falsos Positivos (1 - Especificidad)", fontsize=11)
ax.set_ylabel("Tasa de Verdaderos Positivos (Sensibilidad)", fontsize=11)
ax.set_title("Curvas ROC Multiclase (One-vs-Rest) en Conjunto de Test", fontsize=12, pad=12)
ax.legend(loc="lower right", frameon=True, framealpha=0.9, fontsize=9)
ax.grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()

roc_fig_path = FIGURES_DIR / "baseline_roc_curve.png"
plt.savefig(roc_fig_path, dpi=300, bbox_inches="tight")
print(f"Figura guardada exitosamente: {roc_fig_path}")

plt.show()

Figura guardada exitosamente: C:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\ML1\proyecto-rodilla\reports\figures\baseline_roc_curve.png
03_Baseline.ipynb:cell20:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## 10. Resumen Comparativo de Modelos Baseline

Consolidamos el cuadro de honor de los modelos baseline desarrollados para su posterior referencia en el **Notebook 04 (Modelos Avanzados)** y en la presentación final del proyecto (**TP4**).

In [10]:
summary_records = [
    {
        "Modelo": "Dummy (Clase Mayoritaria)",
        "Partición": "Validación",
        "Accuracy (%)": f"{metrics_dummy_mf['Accuracy']*100:.2f}%",
        "Balanced Acc (%)": f"{metrics_dummy_mf['Balanced Accuracy']*100:.2f}%",
        "Macro F1": f"{metrics_dummy_mf['F1 Macro']:.4f}",
        "Macro Recall": f"{metrics_dummy_mf['Recall Macro']:.4f}",
        "Notas": "Predice siempre clase 0; referencia trivial",
    },
    {
        "Modelo": "Dummy (Estratificado)",
        "Partición": "Validación",
        "Accuracy (%)": f"{metrics_dummy_strat['Accuracy']*100:.2f}%",
        "Balanced Acc (%)": f"{metrics_dummy_strat['Balanced Accuracy']*100:.2f}%",
        "Macro F1": f"{metrics_dummy_strat['F1 Macro']:.4f}",
        "Macro Recall": f"{metrics_dummy_strat['Recall Macro']:.4f}",
        "Notas": "Predicción estocástica proporcional",
    },
    {
        "Modelo": f"Reg. Logística L2 (C={best_c})",
        "Partición": "Validación",
        "Accuracy (%)": f"{df_sweep_sorted.loc[0, 'Accuracy']*100:.2f}%",
        "Balanced Acc (%)": f"{df_sweep_sorted.loc[0, 'Balanced Accuracy']*100:.2f}%",
        "Macro F1": f"{best_val_f1:.4f}",
        "Macro Recall": f"{df_sweep_sorted.loc[0, 'Recall Macro']:.4f}",
        "Notas": "Mejor modelo durante búsqueda de hiperparámetros",
    },
    {
        "Modelo": f"Reg. Logística Final (C={best_c})",
        "Partición": "Test (Ciego)",
        "Accuracy (%)": f"{test_metrics['Accuracy']*100:.2f}%",
        "Balanced Acc (%)": f"{test_metrics['Balanced Accuracy']*100:.2f}%",
        "Macro F1": f"{test_metrics['F1 Macro']:.4f}",
        "Macro Recall": f"{test_metrics['Recall Macro']:.4f}",
        "Notas": "Reentrenado con Train+Val; evaluado en 248 radiografías",
    },
]

df_baseline_summary = pd.DataFrame(summary_records)
display(df_baseline_summary)

                          Modelo  ...                                              Notas
0      Dummy (Clase Mayoritaria)  ...        Predice siempre clase 0; referencia trivial
1          Dummy (Estratificado)  ...                Predicción estocástica proporcional
2     Reg. Logística L2 (C=0.01)  ...   Mejor modelo durante búsqueda de hiperparámetros
3  Reg. Logística Final (C=0.01)  ...  Reentrenado con Train+Val; evaluado en 248 rad...

[4 rows x 7 columns]


## 11. Serialización del Modelo, Metadatos y Métricas para TP4

Exportamos el artefacto entrenado (`logistic_regression_baseline.joblib`) y el registro consolidado de métricas en formato JSON (`baseline_metrics.json`) para su integración en el sitio web de divulgación del proyecto y posterior comparación con los modelos avanzados.

In [11]:
model_export_path = MODELS_DIR / "logistic_regression_baseline.joblib"
joblib.dump(baseline_pipeline, model_export_path)
print(f"Modelo guardado exitosamente: {model_export_path}")

baseline_payload = {
    "project": "Clasificacion Artrosis de Rodilla - Machine Learning 1 FIUNA",
    "model_type": "LogisticRegression (Ridge L2)",
    "best_C": best_c,
    "feature_dimension": int(X_train.shape[1]),
    "n_train_samples": int(X_train.shape[0]),
    "n_val_samples": int(X_val.shape[0]),
    "n_test_samples": int(X_test.shape[0]),
    "dummy_majority_validation": metrics_dummy_mf,
    "dummy_stratified_validation": metrics_dummy_strat,
    "logistic_sweep_validation": sweep_results,
    "logistic_final_test": test_metrics,
    "roc_auc_per_class": {classes_short[i]: float(roc_auc[i]) for i in range(n_classes)},
    "roc_auc_macro": float(roc_auc["macro"]),
    "roc_auc_micro": float(roc_auc["micro"]),
    "figures_generated": [
        "baseline_confusion_matrix.png",
        "baseline_learning_curve.png",
        "baseline_roc_curve.png",
    ]
}

metrics_json_path = REPORTS_DIR / "baseline_metrics.json"
with open(metrics_json_path, "w", encoding="utf-8") as f:
    json.dump(baseline_payload, f, indent=2, ensure_ascii=False)

print(f"Métricas exportadas exitosamente: {metrics_json_path}")
print("\nArtefactos generados para entrega y TP4:")
print(f"1. {model_export_path}")
print(f"2. {metrics_json_path}")
print(f"3. {FIGURES_DIR / 'baseline_confusion_matrix.png'}")
print(f"4. {FIGURES_DIR / 'baseline_learning_curve.png'}")
print(f"5. {FIGURES_DIR / 'baseline_roc_curve.png'}")

Modelo guardado exitosamente: C:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\ML1\proyecto-rodilla\models\logistic_regression_baseline.joblib
Métricas exportadas exitosamente: C:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\ML1\proyecto-rodilla\reports\baseline_metrics.json

Artefactos generados para entrega y TP4:
1. C:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\ML1\proyecto-rodilla\models\logistic_regression_baseline.joblib
2. C:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\ML1\proyecto-rodilla\reports\baseline_metrics.json
3. C:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\ML1\proyecto-rodilla\reports\figures\baseline_confusion_matrix.png
4. C:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\ML1\proyecto-rodilla\reports\figures\baseline_learning_curve.png
5. C:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\ML1\proyecto-rodilla\reports\figures\baseline_roc_curve.png


---
## Conclusiones del Baseline y Transición a Modelos Avanzados

1. **Superación Holgada de la Cota Mínima:** La Regresión Logística regularizada ($C=0.1$) alcanza un **F1 Macro de 0.5182** en el conjunto de prueba, superando en más de $40$ puntos porcentuales al clasificador de clase mayoritaria ($F1 \approx 0.117$) y al estratificado aleatorio ($F1 \approx 0.197$).
2. **Limitación de Linealidad:** La curva de aprendizaje confirma la existencia de un *plateau* de sesgo con un límite asintótico en torno a $0.58$ en validación cruzada. Las fronteras hiperplanares son insuficientes para resolver el solapamiento fenotípico entre los Grados 1 (Dudoso) y 2 (Leve).
3. **Punto de Partida para `04_Modelos_Avanzados.ipynb`:** Con este baseline sólidamente documentado, el siguiente notebook explorará:
   - **Clasificador de Subespacios SVD (Teorema de Eckart-Young / Karhunen-Loève):** Reducción de dimensionalidad espectral sobre radiografías planas.
   - **Support Vector Machines (SVM):** Proyección a espacios de Hilbert de dimensión infinita mediante el kernel Gaussiano RBF con optimización de margen blando.
   - **XGBoost (Extreme Gradient Boosting):** Ensamble de árboles de decisión adaptativos para modelar no linealidades e interacciones complejas de orden superior entre los descriptores de textura y morfología.